# Week05 - Capstone project step schedule exploitation

## Strategy
For Week 5, the policy shifts toward **step-schedule exploitation**.

The idea is to use the accumulated data from the starter `.npy` files plus Weeks 1–4 in `Week05.csv`, identify the strongest region seen so far, and make **small, grid-aligned local edits** rather than large exploratory jumps.

This notebook uses three ingredients:
1. a GP surrogate for tie-breaking (`mean + beta * std`)
2. a **step schedule** that shrinks the local move size as more data accumulates
3. a **human-clean candidate set** built around the current best point and a few recent anchors

This matches a conservative exploitation phase: when a function starts showing a useful region, the next move is a modest refinement rather than a global search.


In [1]:
import itertools
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week05.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
24,4,1,2,0.650,0.650,NaN,NaN,NaN,NaN,NaN,NaN,0.195741
25,4,2,2,0.600,0.600,NaN,NaN,NaN,NaN,NaN,NaN,0.065592
26,4,3,3,0.400,0.400,0.400,NaN,NaN,NaN,NaN,NaN,-0.023175
27,4,4,4,0.405,0.395,0.365,0.445,NaN,NaN,NaN,NaN,0.112351
28,4,5,4,0.930,0.070,0.980,0.980,NaN,NaN,NaN,NaN,3139.315002
29,4,6,5,0.550,0.200,0.650,0.900,0.05,NaN,NaN,NaN,-0.488505
30,4,7,6,0.020,0.220,0.320,0.220,0.44,0.72,NaN,NaN,1.942094
31,4,8,8,0.380,0.070,0.360,0.040,0.80,0.07,0.08,0.06,9.423140


In [3]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, d, rows

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)


In [4]:
def build_candidate_pool(best_x, latest_x, second_x, d, step, grid):
    candidates = []

    # Core anchors.
    candidates.append(round_grid(best_x, grid))
    candidates.append(round_grid((best_x + latest_x) / 2.0, grid))
    candidates.append(round_grid((2 * best_x + latest_x) / 3.0, grid))
    candidates.append(round_grid((best_x + second_x) / 2.0, grid))

    # Axis moves around best point.
    for i in range(d):
        for sign in (-1.0, 1.0):
            x = best_x.copy()
            x[i] += sign * step
            candidates.append(round_grid(x, grid))

    # Small diagonal moves.
    for signs in itertools.product([-1.0, 1.0], repeat=min(d, 3)):
        x = best_x.copy()
        for j, s in enumerate(signs):
            x[j] += s * step
        candidates.append(round_grid(x, grid))

    # Pull a little toward the latest point, but on a shrinking schedule.
    direction = latest_x - best_x
    candidates.append(round_grid(best_x + 0.5 * direction, grid))
    candidates.append(round_grid(best_x + 0.25 * direction, grid))

    # Remove duplicates.
    candidates = np.unique(np.array(candidates), axis=0)
    return candidates


In [5]:
def suggest_week5_point(function_id, history_df, *, seed=300):
    X, y, d, rows = load_combined(function_id, history_df)

    best_idx = int(np.argmax(y))
    second_idx = int(np.argsort(y)[-2])
    best_x = X[best_idx].copy()
    second_x = X[second_idx].copy()
    latest_x = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)[-1].copy()

    # Step schedule: smaller edits as d grows.
    base_steps = {2: 0.03, 3: 0.05, 4: 0.01, 5: 0.03, 6: 0.01, 8: 0.06}
    step = base_steps.get(d, 0.02)
    grid = 0.01

    candidates = build_candidate_pool(best_x, latest_x, second_x, d, step, grid)

    # Remove points already tested.
    dist = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)
    keep = min_dist >= 0.005
    candidates = candidates[keep]
    min_dist = min_dist[keep]

    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.05, 2.0))
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-2))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed + function_id,
        n_restarts_optimizer=1,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)

    mean, std = gp.predict(candidates, return_std=True)
    best_dist = np.sqrt(((candidates - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((candidates - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)

    # Conservative exploitation score.
    score = mean + 0.8 * std - 0.25 * best_dist - 0.10 * latest_dist + 0.05 * min_dist
    best_candidate_idx = int(np.argmax(score))
    x_best = candidates[best_candidate_idx]

    return {
        'function': function_id,
        'd': d,
        'x': x_best,
        'formatted': format_point(x_best),
        'step': step,
        'mean_pred': float(mean[best_candidate_idx]),
        'std_pred': float(std[best_candidate_idx]),
        'min_dist': float(min_dist[best_candidate_idx]),
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
        'kernel': str(gp.kernel_),
    }


In [ ]:
results = []
for function_id in range(1, 9):
    result = suggest_week5_point(function_id, history)
    results.append(result)
    print(f'Function {function_id}: {result["formatted"]}')
    print(f'  step={result["step"]:.3f}, mean={result["mean_pred"]:.6f}, std={result["std_pred"]:.6f}, min_dist={result["min_dist"]:.6f}')
    print(f'  best_anchor={result["best_anchor"]}')
    print(f'  latest_anchor={result["latest_anchor"]}')
    print()
